# Train a model for the NAT detector

This small, runnable example shows the model-independent training contract. Replace the example estimator, aggregation, and flow table with a real experiment. Training and execution both call `NATClassifier.aggregate()` on one IP's flows from one time window.

In [ ]:
from pathlib import Path
import sys

import joblib
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, recall_score
from sklearn.model_selection import StratifiedGroupKFold

repo_root = next(
    parent
    for parent in (Path.cwd(), *Path.cwd().parents)
    if (parent / "pyproject.toml").exists()
)
if str(repo_root / "src") not in sys.path:
    sys.path.insert(0, str(repo_root / "src"))

from detectors.common import TimeWindowedIPFlowDataset
from detectors.nat_detector import NATClassifier

## 1. Load labeled flows

The example data keeps the notebook runnable. In a real experiment, replace this cell with `pd.read_csv(...)`. Each row needs a source IP, timestamp, raw columns used by the aggregation, and a binary label.

In [ ]:
rows = []
start = pd.Timestamp("2026-01-01T10:00:00Z")
for ip_index in range(40):
    is_nat = ip_index % 2
    for flow_index in range(6):
        rows.append(
            {
                "SRC_IP": f"192.0.2.{ip_index + 1}",
                "TIME_FIRST": start + pd.Timedelta(minutes=flow_index),
                "IP_TTL": 64 + is_nat * (flow_index % 3),
                "DST_PORT": 443 if not is_nat else 1000 + flow_index,
                "IS_NAT": is_nat,
            }
        )
flows = pd.DataFrame(rows)
flows.head()

## 2. Choose an estimator and aggregation

The estimator may be replaced by any fitted-compatible classifier implementing `fit()` and `predict_proba()`. Aggregation entries are pandas named aggregations and become the estimator's ordered feature columns.

In [ ]:
aggregation = {
    "unique_ttl_values": ("IP_TTL", "nunique"),
    "unique_destination_ports": ("DST_PORT", "nunique"),
}
classifier = NATClassifier(
    model=LogisticRegression(random_state=42),
    aggregation=aggregation,
    positive_label=1,
    threshold=0.5,
)
print("Required raw columns:", classifier.required_columns)
print("Ordered model features:", classifier.feature_names)

## 3. Build IP/window samples

Use the same window size here and under `daf.windowing` in production. Labels are checked separately so they never become model inputs.

In [ ]:
dataset = TimeWindowedIPFlowDataset(
    flows,
    timestamp_column="TIME_FIRST",
    window_size="15min",
)
X = dataset.apply(classifier.aggregate)
label_summary = pd.concat(
    {
        window_start: ip_dataset.agg(
            is_nat=("IS_NAT", "first"),
            distinct_labels=("IS_NAT", "nunique"),
        )
        for window_start, ip_dataset in dataset
    },
    names=["window_start", "SRC_IP"],
)
if not label_summary["distinct_labels"].eq(1).all():
    raise ValueError("An IP/window contains conflicting labels")
y = label_summary["is_nat"].reindex(X.index)

## 4. Split, train, and evaluate

Grouping by source IP prevents different windows for one IP from leaking across the split.

In [ ]:
groups = X.index.get_level_values("SRC_IP")
splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
train_positions, test_positions = next(splitter.split(X, y, groups))
X_train, X_test = X.iloc[train_positions], X.iloc[test_positions]
y_train, y_test = y.iloc[train_positions], y.iloc[test_positions]

classifier.model.fit(X_train, y_train)
predictions = classifier.model.predict(X_test)
pd.Series(
    {
        "accuracy": accuracy_score(y_test, predictions),
        "f1": f1_score(y_test, predictions),
        "recall": recall_score(y_test, predictions),
    }
)

## 5. Save the complete classifier

DAF must load the `NATClassifier`, not the bare estimator. Choose a model package under `nat_detector/final_models`, document its raw-column mapping, then enable these lines.

In [ ]:
# model_path = (
#     repo_root / "src/detectors/nat_detector/final_models/my_model/nat_classifier.joblib"
# )
# model_path.parent.mkdir(parents=True, exist_ok=True)
# joblib.dump(classifier, model_path)
# loaded_classifier = joblib.load(model_path)
# isinstance(loaded_classifier, NATClassifier)